# Assignment: Representation Learning using PCA and Autoencoders
Complete end-to-end notebook for MNIST and CIFAR-10.

## 1. Imports

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import mnist, cifar10
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_curve, auc
)

np.random.seed(42)
tf.random.set_seed(42)


## 2. Dataset Preprocessing

In [ ]:

TARGET_SIZE = 28

def normalize_50_200(x):
    x = x.astype(np.float32)
    mn = x.min()
    mx = x.max()
    return 50 + ((x - mn)/(mx-mn))*150

def load_mnist():
    (xtr,ytr),(xte,yte)=mnist.load_data()
    X=np.concatenate([xtr,xte])
    y=np.concatenate([ytr,yte])
    X=normalize_50_200(X)
    return X,y

def load_cifar():
    (xtr,ytr),(xte,yte)=cifar10.load_data()
    X=np.concatenate([xtr,xte])
    y=np.concatenate([ytr,yte]).ravel()

    gray=np.dot(X[...,:3],[0.2989,0.5870,0.1140])

    resized=np.array([
        tf.image.resize(img[...,None],[28,28]).numpy().squeeze()
        for img in gray
    ])

    resized=normalize_50_200(resized)
    return resized,y

mnist_X,mnist_y=load_mnist()
cifar_X,cifar_y=load_cifar()

print(mnist_X.shape)
print(cifar_X.shape)


## 3. 70/20/10 Split

In [ ]:

def create_split(X,y):

    X_train,X_temp,y_train,y_temp=train_test_split(
        X,y,test_size=0.30,
        random_state=42,
        stratify=y
    )

    X_val,X_test,y_val,y_test=train_test_split(
        X_temp,y_temp,
        test_size=1/3,
        random_state=42,
        stratify=y_temp
    )

    return X_train,X_val,X_test,y_train,y_val,y_test


## 4. Utility Functions

In [ ]:

def flatten(X):
    return X.reshape(len(X),-1)

def compute_snr(original,reconstructed):

    signal=np.mean(original**2,axis=1)
    noise=np.mean((original-reconstructed)**2,axis=1)+1e-12

    return np.mean(
        10*np.log10(signal/noise)
    )

def multiclass_roc(y_true,probs,title):

    y_bin=label_binarize(y_true,classes=np.arange(10))

    plt.figure(figsize=(8,6))

    for i in range(10):
        fpr,tpr,_=roc_curve(y_bin[:,i],probs[:,i])
        roc_auc=auc(fpr,tpr)
        plt.plot(fpr,tpr,label=f"{i}: {roc_auc:.3f}")

    plt.plot([0,1],[0,1],'k--')
    plt.title(title)
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()
    plt.show()


## Task 1 - Standard PCA

In [ ]:

def run_standard_pca(X,y,name):

    X_train,X_val,X_test,y_train,y_val,y_test=create_split(X,y)

    Xtr=flatten(X_train)
    Xte=flatten(X_test)

    pca=PCA(
        n_components=30,
        svd_solver='full'
    )

    Ztr=pca.fit_transform(Xtr)
    Zte=pca.transform(Xte)

    clf=LogisticRegression(
        max_iter=3000,
        n_jobs=-1
    )

    clf.fit(Ztr,y_train)

    preds=clf.predict(Zte)
    probs=clf.predict_proba(Zte)

    acc=accuracy_score(y_test,preds)

    recon=pca.inverse_transform(Zte)

    snr=compute_snr(Xte,recon)

    multiclass_roc(
        y_test,
        probs,
        f'{name} Standard PCA ROC'
    )

    return pca,acc,snr


## Task 1 - Randomized PCA

In [ ]:

def run_randomized_pca(X,y,name):

    X_train,X_val,X_test,y_train,y_val,y_test=create_split(X,y)

    Xtr=flatten(X_train)
    Xte=flatten(X_test)

    rpca=PCA(
        n_components=30,
        svd_solver='randomized'
    )

    Ztr=rpca.fit_transform(Xtr)
    Zte=rpca.transform(Xte)

    clf=LogisticRegression(
        max_iter=3000,
        n_jobs=-1
    )

    clf.fit(Ztr,y_train)

    preds=clf.predict(Zte)
    probs=clf.predict_proba(Zte)

    acc=accuracy_score(y_test,preds)

    recon=rpca.inverse_transform(Zte)

    snr=compute_snr(Xte,recon)

    multiclass_roc(
        y_test,
        probs,
        f'{name} Randomized PCA ROC'
    )

    return rpca,acc,snr


## Run PCA Experiments

In [ ]:

mnist_pca,mnist_pca_acc,mnist_pca_snr=run_standard_pca(
    mnist_X,mnist_y,'MNIST'
)

mnist_rpca,mnist_rpca_acc,mnist_rpca_snr=run_randomized_pca(
    mnist_X,mnist_y,'MNIST'
)

cifar_pca,cifar_pca_acc,cifar_pca_snr=run_standard_pca(
    cifar_X,cifar_y,'CIFAR10'
)

cifar_rpca,cifar_rpca_acc,cifar_rpca_snr=run_randomized_pca(
    cifar_X,cifar_y,'CIFAR10'
)


## Display PCA Eigenvectors

In [ ]:

def show_eigenvectors(pca,title):

    plt.figure(figsize=(15,10))

    for i in range(30):
        plt.subplot(5,6,i+1)
        plt.imshow(
            pca.components_[i].reshape(28,28),
            cmap='gray'
        )
        plt.axis('off')

    plt.suptitle(title)
    plt.show()

show_eigenvectors(
    mnist_pca,
    'MNIST PCA Eigenvectors'
)

show_eigenvectors(
    cifar_pca,
    'CIFAR PCA Eigenvectors'
)


## Task 2 - Linear Autoencoder

In [ ]:

def build_linear_autoencoder(input_dim):

    inp=Input(shape=(input_dim,))

    latent=Dense(
        30,
        activation='linear',
        use_bias=False
    )(inp)

    out=Dense(
        input_dim,
        activation='linear',
        use_bias=False
    )(latent)

    autoencoder=Model(inp,out)

    encoder=Model(inp,latent)

    autoencoder.compile(
        optimizer='adam',
        loss='mse'
    )

    return autoencoder,encoder


In [ ]:

def train_linear_ae(X,y,pca_obj,name):

    X_train,X_val,X_test,y_train,y_val,y_test=create_split(X,y)

    scaler=StandardScaler()

    Xtr=scaler.fit_transform(flatten(X_train))
    Xte=scaler.transform(flatten(X_test))

    ae,encoder=build_linear_autoencoder(
        Xtr.shape[1]
    )

    ae.fit(
        Xtr,Xtr,
        epochs=30,
        batch_size=256,
        validation_split=0.1,
        verbose=1
    )

    Ftr=encoder.predict(Xtr)
    Fte=encoder.predict(Xte)

    clf=LogisticRegression(
        max_iter=3000
    )

    clf.fit(Ftr,y_train)

    acc=clf.score(Fte,y_test)

    weights=encoder.layers[-1].get_weights()[0]

    pca_vecs=pca_obj.components_.T

    sims=[]

    for i in range(30):
        a=pca_vecs[:,i]
        b=weights[:,i]

        sims.append(
            abs(
                np.dot(a,b)/(
                    np.linalg.norm(a)*
                    np.linalg.norm(b)
                )
            )
        )

    similarity=np.mean(sims)

    return ae,encoder,acc,similarity


In [ ]:

mn_ae,mn_encoder,mn_ae_acc,mn_similarity=train_linear_ae(
    mnist_X,
    mnist_y,
    mnist_pca,
    'MNIST'
)

cf_ae,cf_encoder,cf_ae_acc,cf_similarity=train_linear_ae(
    cifar_X,
    cifar_y,
    cifar_pca,
    'CIFAR10'
)


## Visualize Autoencoder Weights

In [ ]:

def show_weights(weights,title):

    plt.figure(figsize=(15,10))

    for i in range(30):
        plt.subplot(5,6,i+1)

        plt.imshow(
            weights[:,i].reshape(28,28),
            cmap='gray'
        )

        plt.axis('off')

    plt.suptitle(title)
    plt.show()

show_weights(
    mn_encoder.layers[-1].get_weights()[0],
    'MNIST AE Weights'
)

show_weights(
    cf_encoder.layers[-1].get_weights()[0],
    'CIFAR AE Weights'
)


## Task 3 - Convolutional Autoencoder

In [ ]:

def build_conv_autoencoder():

    inp=Input((28,28,1))

    x=Conv2D(32,3,padding='same',
             activation='relu')(inp)
    x=MaxPooling2D()(x)

    x=Conv2D(16,3,padding='same',
             activation='relu')(x)
    x=MaxPooling2D()(x)

    x=Flatten()(x)

    latent=Dense(30,name='latent')(x)

    x=Dense(7*7*16,
            activation='relu')(latent)

    x=Reshape((7,7,16))(x)

    x=UpSampling2D()(x)

    x=Conv2D(16,3,padding='same',
             activation='relu')(x)

    x=UpSampling2D()(x)

    out=Conv2D(
        1,
        3,
        padding='same',
        activation='linear'
    )(x)

    model=Model(inp,out)

    model.compile(
        optimizer='adam',
        loss='mse'
    )

    return model


## Three Hidden Layer Autoencoder

In [ ]:

def build_three_layer_autoencoder(input_dim):

    inp=Input(shape=(input_dim,))

    x=Dense(10,activation='sigmoid')(inp)
    x=Dense(10,activation='sigmoid')(x)
    x=Dense(10,activation='sigmoid')(x)

    x=Dense(10,activation='sigmoid')(x)
    x=Dense(10,activation='sigmoid')(x)

    out=Dense(
        input_dim,
        activation='linear'
    )(x)

    model=Model(inp,out)

    model.compile(
        optimizer='adam',
        loss='mse'
    )

    return model


## Evaluate Task 3 Models

In [ ]:

def evaluate_conv_autoencoder(X):

    Xtr,Xval,Xte,_,_,_=create_split(
        X,
        np.zeros(len(X))
    )

    model=build_conv_autoencoder()

    model.fit(
        Xtr[...,None],
        Xtr[...,None],
        validation_data=(
            Xval[...,None],
            Xval[...,None]
        ),
        epochs=20,
        batch_size=256
    )

    recon=model.predict(
        Xte[...,None]
    )

    snr=compute_snr(
        flatten(Xte),
        flatten(recon.squeeze())
    )

    return snr

mnist_conv_snr = evaluate_conv_autoencoder(
    mnist_X
)

cifar_conv_snr = evaluate_conv_autoencoder(
    cifar_X
)


In [ ]:

def evaluate_three_layer_ae(X):

    Xtr,Xval,Xte,_,_,_=create_split(
        X,
        np.zeros(len(X))
    )

    scaler=StandardScaler()

    Xtr=scaler.fit_transform(
        flatten(Xtr)
    )

    Xte=scaler.transform(
        flatten(Xte)
    )

    model=build_three_layer_autoencoder(
        784
    )

    model.fit(
        Xtr,Xtr,
        epochs=30,
        batch_size=256
    )

    recon=model.predict(Xte)

    return compute_snr(Xte,recon)

mnist_3ae_snr=evaluate_three_layer_ae(mnist_X)

cifar_3ae_snr=evaluate_three_layer_ae(cifar_X)


## Final Comparison Table

In [ ]:

results=pd.DataFrame({

'Dataset':[
'MNIST',
'CIFAR10'
],

'PCA Accuracy':[
mnist_pca_acc,
cifar_pca_acc
],

'Random PCA Accuracy':[
mnist_rpca_acc,
cifar_rpca_acc
],

'Linear AE Accuracy':[
mn_ae_acc,
cf_ae_acc
],

'PCA SNR':[
mnist_pca_snr,
cifar_pca_snr
],

'Random PCA SNR':[
mnist_rpca_snr,
cifar_rpca_snr
],

'Conv AE SNR':[
mnist_conv_snr,
cifar_conv_snr
],

'3 Layer AE SNR':[
mnist_3ae_snr,
cifar_3ae_snr
],

'PCA-AE Similarity':[
mn_similarity,
cf_similarity
]

})

results


## Conclusions
1. Compare PCA vs Randomized PCA accuracy and SNR.
2. Compare PCA eigenvectors with linear AE weights using cosine similarity.
3. Compare ConvAE and 3-layer AE SNR.
4. State which representation method performs best for each dataset.